# Day 33: CNN Basics & Transfer Learning Exercises

Welcome to **Day 33**! In these exercises, we will explore the foundational building blocks of Computer Vision in PyTorch:
1. **Convolution Layers (`nn.Conv2d`)**: How sliding filters detect spatial patterns.
2. **Pooling Layers (`nn.MaxPool2d`)**: How to downsample feature maps while keeping dominant signals.
3. **Image Transforms (`torchvision.transforms`)**: Preparing raw images for neural networks.
4. **Pretrained Models & Freezing**: Loading ResNet-18, freezing its feature extractor, and replacing the final classification head.

### Exercise 1: Exploring `nn.Conv2d` and `nn.MaxPool2d`
- Input image shape in PyTorch: `(Batch_Size, Channels, Height, Width)` $\to$ `(N, C, H, W)`.
- Let's create a dummy RGB image tensor `(1, 3, 32, 32)` and trace how its shape changes after convolution and pooling.

In [1]:
import torch
import torch.nn as nn

# 1. Create a dummy batch of 1 color image (3 channels, 32x32 pixels)
x = torch.randn(1, 3, 32, 32)
print(f"Initial Input Shape (N, C, H, W): {x.shape}")

# 2. Convolution Layer: 3 input channels -> 16 output channels, 3x3 filter, padding=1 (keeps H, W same)
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
out_conv = conv(x)
print(f"After Conv2d (16 filters):       {out_conv.shape}")

# 3. Max Pooling Layer: 2x2 window with stride 2 (cuts H and W in half)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
out_pool = pool(out_conv)
print(f"After MaxPool2d (downsampled):   {out_pool.shape}")

Initial Input Shape (N, C, H, W): torch.Size([1, 3, 32, 32])
After Conv2d (16 filters):       torch.Size([1, 16, 32, 32])
After MaxPool2d (downsampled):   torch.Size([1, 16, 16, 16])


### Exercise 2: Image Transforms & Augmentation
Pretrained models like ResNet expect input images to be:
1. Converted to a float tensor in range `[0.0, 1.0]`.
2. Resized to at least `224x224` pixels (or matching target architecture).
3. Normalized using ImageNet's standard mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`.

In [2]:
from torchvision import transforms

# Define standard preprocessing pipeline for transfer learning
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transform pipeline defined successfully:")
print(transform_pipeline)

Transform pipeline defined successfully:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


### Exercise 3 & 4: Loading Pretrained ResNet-18, Freezing, and Swapping the Head
- Load `resnet18` from `torchvision.models`.
- Freeze all backbone parameters by setting `param.requires_grad = False`.
- Inspect the original `model.fc` (outputs 1000 classes for ImageNet).
- Replace `model.fc` with a new `nn.Linear` layer that outputs our desired number of classes (e.g., 10 for CIFAR-10).

In [3]:
import torchvision.models as models
from torchvision.models import ResNet18_Weights

# 1. Load pretrained ResNet-18
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

# 2. Freeze all backbone layers
for param in model.parameters():
    param.requires_grad = False

# 3. Check original fully-connected (fc) head
in_features = model.fc.in_features
print(f"Original ResNet fc input features: {in_features}, output features: {model.fc.out_features}")

# 4. Replace with new head for 10 classes (requires_grad is True by default for new layers)
num_classes = 10
model.fc = nn.Linear(in_features, num_classes)
print(f"New custom fc layer: {model.fc}")

# 5. Verify parameter trainable status
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({trainable_params/total_params:.2%})")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Kavya/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:12<00:00, 3.84MB/s]


Original ResNet fc input features: 512, output features: 1000
New custom fc layer: Linear(in_features=512, out_features=10, bias=True)
Trainable parameters: 5,130 / 11,181,642 (0.05%)
